In [ ]:
#import sys
#!{sys.executable} -m pip install --upgrade c-lasso 
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
from os.path import join, dirname, abspath

# this file is in classo/examples/scripts, so we need to go up three levels to get the classo directory
classo_dir = dirname(dirname(os.getcwd()))
sys.path.append(classo_dir)

figure_dir = join(classo_dir, "figures")

from classo import clr, classo_problem
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

In [ ]:
def csv_to_np(file, begin = 1, header = None):
    """Function to read a csv file and to create an ndarray with this

    Args:
        file (str): Name of csv file
        begin (int, optional): First colomn where it should read the matrix
        header (None or int, optional): Same parameter as in the function :func:`pandas.read_csv`

    Returns:
        ndarray : matrix of the csv file
    """
    tab1 = pd.read_csv(file, header = header)
    return np.array(tab1)[:, begin:]

# Combo data

Combettes, P. L. and Müller, C. L. (2020). Regression models for compositional data: Gen- eral log-contrast formulations, proximal optimization, and microbiome data applications. Statistics in Biosciences, pages 1–26.:

"The data set comprises the log-transformed relative abundances of p = 87 taxa at the genus level across n = 96 healthy subjects. Following previous analyses, we investigate the relationship between BMI and the microbial compositions in a log-contrast regression framework."

## Building matrices 

In [ ]:
figure_dir_COMBO = join(figure_dir, 'exampleCOMBO')
data_dir_COMBO = join(classo_dir, 'examples', 'datasets', 'COMBO_data')
X0  = csv_to_np(join(data_dir_COMBO, 'complete_data/GeneraCounts.csv'), begin = 0).astype(float)
X_C = csv_to_np(join(data_dir_COMBO, 'CaloriData.csv'), begin = 0).astype(float)
X_F = csv_to_np(join(data_dir_COMBO, 'FatData.csv'), begin = 0).astype(float)
y   = csv_to_np(join(data_dir_COMBO, 'BMI.csv'), begin = 0).astype(float)[:, 0]
labels = csv_to_np(join(data_dir_COMBO, 'complete_data/GeneraPhylo.csv')).astype(str)[:, -1]

y   = y - np.mean(y) #BMI data (n = 96)
X_C = X_C - np.mean(X_C, axis = 0)  #Covariate data (Calorie)
X_F = X_F - np.mean(X_F, axis = 0)  #Covariate data (Fat)
X0 = clr(X0, 1 / 2).T

X     = np.concatenate((X0, X_C, X_F), axis = 1) # Joint microbiome and covariate data and offset
label = np.concatenate([labels, np.array(['Calorie', 'Fat'])])
C = np.ones((1, len(X[0])))
C[0, -1], C[0, -2] = 0., 0.

X.shape

## Create classo instance and change parameters 

In [ ]:
problem = classo_problem(X, y, C, label = label)

#problem.formulation.intercept = True
problem.model_selection.LAMfixed                    = False
problem.model_selection.StabSelparameters.method      = 'lam'
problem.model_selection.StabSelparameters.threshold_label = 0.5
problem.model_selection.PATH = True
# problem.model_selection.ALO = True

## R1

In [ ]:
problem.data.label = label
problem.formulation.concomitant = False
problem.solve()
problem.solution.PATH.save = f'{figure_dir_COMBO}/R1-'
problem.solution.StabSel.save1 = f'{figure_dir_COMBO}/R1-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_COMBO}/R1-StabSel-beta'
print(problem, problem.solution)

## R2

In [ ]:
problem.data.label = label
problem.formulation.huber = True
problem.solve()

problem.solution.PATH.save = f'{figure_dir_COMBO}/R2-'
problem.solution.StabSel.save1 = f'{figure_dir_COMBO}/R2-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_COMBO}/R2-StabSel-beta'
print(problem, problem.solution)

## R3

In [ ]:
problem.data.label = label
problem.formulation.concomitant = True
problem.formulation.huber = False


problem.solve()
problem.solution.PATH.save = f'{figure_dir_COMBO}/R3-'
problem.solution.StabSel.save1 = f'{figure_dir_COMBO}/R3-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_COMBO}/R3-StabSel-beta'
print(problem, problem.solution)

## R4

In [ ]:
problem.formulation.huber = True
problem.model_selection.StabSelparameters.seed = 123
problem.solve()
problem.solution.PATH.save = f'{figure_dir_COMBO}/R4-'
problem.solution.StabSel.save1 = f'{figure_dir_COMBO}/R4-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_COMBO}/R4-StabSel-beta'
print(problem, problem.solution)

# Filtered Combo data

In [ ]:
figure_dir_FilteredCOMBO = join(figure_dir, 'exampleFilteredCOMBO')


# Load microbiome and covariate data X
X_C = csv_to_np(join(data_dir_COMBO, 'CaloriData.csv'), begin = 0).astype(float)
X_F = csv_to_np(join(data_dir_COMBO, 'FatData.csv'), begin = 0).astype(float)
X0  = csv_to_np(join(data_dir_COMBO, 'filtered_data/GeneraFilteredCounts.csv'), begin = 0).astype(float)


# Load BMI measurements y
y   = csv_to_np(join(data_dir_COMBO, 'BMI.csv'), begin = 0).astype(float)[:, 0]

# Load genus and covariate labels
labels  = csv_to_np(join(data_dir_COMBO, 'filtered_data/GeneraFilteredPhylo.csv')).astype(str)[:, -1]


# Normalize/transform data
y   = y - np.mean(y) #BMI data (n = 96)
X_C = X_C - np.mean(X_C, axis = 0)  #Covariate data (Calorie)
X_F = X_F - np.mean(X_F, axis = 0)  #Covariate data (Fat)
X0 = clr(X0, 1 / 2).T

# Set up design matrix and zero-sum constraints for 45 genera
X      = np.concatenate((X0, X_C, X_F, np.ones((len(X0), 1))), axis = 1) # Joint microbiome and covariate data and offset
label = np.concatenate([labels,np.array(['Calorie', 'Fat', 'Bias'])])
C = np.ones((1,len(X[0])))
C[0, -1], C[0, -2], C[0, -3] = 0., 0., 0.

# Set up c-lassso problem
problem = classo_problem(X, y, C, label = label)

# Use stability selection with theoretical lambda [Combettes & Müller, 2020b]
problem.model_selection.StabSel                       = True
problem.model_selection.StabSelparameters.method      = 'lam'
problem.model_selection.StabSelparameters.seed     = 2


In [ ]:
# Use formulation R3
problem.formulation.concomitant = True

problem.solve()

In [ ]:
problem.solution.StabSel.save1 = f'{figure_dir_FilteredCOMBO}/R3-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_FilteredCOMBO}/R3-StabSel-beta'

In [ ]:
print(problem)
print(problem.solution)

In [ ]:
# Use formulation R4
problem.formulation.huber = True
problem.formulation.concomitant = True

problem.solve()

In [ ]:
problem.solution.StabSel.save1 = f'{figure_dir_FilteredCOMBO}/R4-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_FilteredCOMBO}/R4-StabSel-beta'

In [ ]:
print(problem)
print(problem.solution)

# pH data

## Building matrices 

In [ ]:
figure_dir_pH = join(figure_dir, 'examplepH')

data_dir_pH = join(classo_dir, 'examples', 'datasets', 'pH_data')
pH = sio.loadmat(join(data_dir_pH, 'matlab', 'pHData.mat'))
tax = sio.loadmat(join(data_dir_pH, 'matlab', 'taxTablepHData.mat'))

X, Y_uncent = pH['X'], pH['Y'].T[0]
y = Y_uncent - np.mean(Y_uncent) # Center Y

X.shape

## Create classo instance and change parameters 

In [ ]:
problem = classo_problem(X, y)

#problem.model_selection.LAMfixed                    = True
problem.model_selection.StabSelparameters.method      = 'lam'
problem.model_selection.PATH = True
problem.model_selection.LAMfixed = True

## R1

In [ ]:
problem.formulation.concomitant = False
problem.solve()
problem.solution.PATH.save = f'{figure_dir_pH}/R1-'
problem.solution.StabSel.save1 = f'{figure_dir_pH}/R1-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_pH}/R1-StabSel-beta'
problem.solution.LAMfixed.save = f'{figure_dir_pH}/R1-beta'
print(problem, problem.solution)

## R2

In [ ]:
problem.formulation.huber = True
problem.solve()
print(problem, problem.solution)

## R3

In [ ]:
problem.formulation.concomitant = True
problem.formulation.huber = False

problem.solve()
problem.solution.PATH.save = f'{figure_dir_pH}/R3-'
problem.solution.StabSel.save1 = f'{figure_dir_pH}/R3-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_pH}/R3-StabSel-beta'
problem.solution.LAMfixed.save = f'{figure_dir_pH}/R3-beta'

print(problem, problem.solution)

## R4

In [ ]:
problem.formulation.huber = True
problem.solve()
print(problem, problem.solution)

# Tara data

Bien, J., Yan, X., Simpson, L. and Müller, C. (2020). Tree-Aggregated Predictive Modeling of Microbiome Data : 

"Integrative marine data collection efforts such as Tara Oceans (Sunagawa et al., 2020) or the Simons CMAP (https://simonscmap.com) provide the means to investigate ocean ecosystems on a global scale. Using Tara’s environmental and microbial survey of ocean surface water (Sunagawa, 2015), we next illustrate how trac can be used to globally connect environmental covariates and the ocean microbiome. As an example, we learn a global predictive model of ocean salinity from n = 136 samples and p = 8916 miTAG OTUs (Logares et al., 2014). trac identifies four taxonomic aggregations, the kingdom bacteria and the phylum Bacteroidetes being negatively associated and the classes Alpha and Gammaproteobacteria being positively associated with marine salinity (see Figure 7 lower panel). Figure 7 shows the scatter plot of salinity measurements vs. trac model predictions (with correlation on the out-of-sample test set of 0.55)."

## Building matrices 

###  importing data

In [ ]:
# import sys
# !{sys.executable} -m pip install rdata

In [ ]:
figure_dir_Tara = join(figure_dir, 'exampleTara')
data_dir_Tara = join(classo_dir, 'examples', 'datasets', 'Tara')
import warnings
import rdata
from scipy.sparse import csc_matrix

Open R file tara_sal_processed.RDS

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Missing constructor for R class")
    rds = rdata.read_rds(join(data_dir_Tara, "tara_sal_processed.RDS"))

In [ ]:
x = rds["x"].values
y = rds["y"]
# R's dgCMatrix slots use the same CSC layout as SciPy.
rA = rds["A"]
A = csc_matrix((rA.x, rA.i, rA.p), shape=tuple(rA.Dim)).toarray()

In [ ]:
label_OTU = rds["x"].coords[rds["x"].dims[1]].values
label_sample = rds["x"].coords[rds["x"].dims[0]].values
label_nodes = np.asarray(rA.Dimnames[1], dtype=str)
label_short = [l.split("::")[-1] for l in label_nodes]
print(y.shape)
print(x.shape)
print(A.shape)

### process similar to 2fit_trac_model.R

In [ ]:
# Load the saved training split and cross-validation results.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Missing constructor for R class")
    workspace = rdata.read_rda(join(data_dir_Tara, "tara_sal_trac.Rdata"))

In [ ]:
cvfit = workspace["cvfit"]
# The bundled fit has a single cross-validation entry.
cv = cvfit["cv"][0]
lambda_1SE = cv["lambda_1se"][0]

In [ ]:
lambda_1SE

In [ ]:
tr = workspace["tr"] - 1  # Convert R's one-based indices to Python.
te = np.array([i for i in range(len(y)) if not i in tr])

In [ ]:
len(tr)

### process similar to trac 

In [ ]:
pseudo_count = 1
X = np.log(pseudo_count+x)
nleaves = np.sum(A,axis = 0)
logGeom = X.dot(A)/nleaves

## Create classo instance and change parameters 

In [ ]:
problem = classo_problem(logGeom[tr], y[tr], label = label_short)

problem.formulation.w = 1/nleaves
problem.formulation.intercept     = True
problem.formulation.concomitant = False

## Cross validation and Path Computation

### Computation 

In [ ]:
problem.model_selection.StabSel   = False
problem.model_selection.PATH   = True
problem.model_selection.CV   = True
problem.model_selection.CVparameters.seed = 6 # one could change logscale, Nsubset, oneSE
problem

In [ ]:
problem.solve()

In [ ]:
problem.solution.CV.save = f'{figure_dir_pH}/R1-CVbeta'
problem.solution.PATH.save = f'{figure_dir_pH}/R1-Path'

In [ ]:
len(problem.data.label)

In [ ]:
print(problem.solution)

In [ ]:
problem.solution.CV.graphic(logscale = True, save = f'{figure_dir_pH}/R1-CV')

In [ ]:
print("lambda_1SE : ", problem.solution.CV.lambda_1SE)
print("lambda_best : ", problem.solution.CV.lambda_min)

In [ ]:
selection = problem.solution.CV.selected_param[1:] # exclude the intercept
label_nodes[selection]

### Prediction plot

In [ ]:
alpha = problem.solution.CV.refit
yhat = logGeom[te].dot(alpha[1:])+alpha[0]

In [ ]:
M1, M2 = max(y[te]), min(y[te])
plt.plot(yhat, y[te], 'bo', label = 'sample of the testing set')
plt.plot([M1, M2], [M1, M2], 'k-', label = "identity")
plt.xlabel('predictor yhat'), plt.ylabel('real y'), plt.legend()
plt.show()

## Stability Selection

### Computation 

In [ ]:
problem.model_selection.StabSel   = True
problem.model_selection.PATH   = False
problem.model_selection.CV   = False
# can change q, B, nS, method, threshold etc in problem.model_selection.StabSelparameters
problem

In [ ]:
problem.solve()

In [ ]:
problem.solution.StabSel.save1 = f'{figure_dir_pH}/R1-StabSel'
problem.solution.StabSel.save2 = f'{figure_dir_pH}/R1-StabSel-beta'

In [ ]:
problem.solution

In [ ]:
selection = problem.solution.StabSel.selected_param[1:] # exclude the intercept
label_nodes[selection]

### Prediction plot

In [ ]:
alpha = problem.solution.StabSel.refit
yhat = logGeom[te].dot(alpha[1:])+alpha[0]

In [ ]:
M1, M2 = max(y[te]), min(y[te])
plt.plot(yhat, y[te], 'bo', label = 'sample of the testing set')
plt.plot([M1, M2],[M1, M2], 'k-', label = "identity")
plt.xlabel('predictor yhat'), plt.ylabel('real y'), plt.legend()
plt.show()